# C.3 v6 — D=8192 confirmation + D=16384 extension

**Why this notebook.** v5 D-sweep produced a striking result:
- At D=4096, three independent n=10 seed groups gave Δ ∈ {+0.055, +0.023, −0.017} (default/spread); pooled n=30 is borderline-disjoint in one stratum only.
- At D=8192 with seeds 30..39, both strata are CI-disjoint at the v3-graduation magnitude (Δ +0.057 default/spread, Δ +0.081 calibrated/tight).
- The same seeds 30..39 had NEGATIVE Δ at D=4096 but POSITIVE at D=8192 — suggesting the effect is substrate-dim-dependent, not seed-lucky.

v6 confirms or falsifies this with two probes:

| probe | β | D | seeds | role |
|---|---:|---:|---|---|
| `D8192_confirm_seeds0_19` | 10 | 8192 | 0..19 | 20 new procs → pool with v5 D=8192 seeds 30..39 → n=30 at D=8192 |
| `D16384_seeds30_39` | 10 | 16384 | 30..39 | curve extension: does effect grow or saturate at 16k? |

All else fixed at the v3 config: `wikitext, lr_pull=0.1, n_events=1000, alpha_anti=0.01, repulsion_step_size=0.05, window=8, vocab_cap=1000`.

**30 parallel CUDA subprocesses.** ~50-60 GB GPU peak (D=16384 procs are large).

**Aggregation produces three outputs:**
1. **Pooled n=30 at D=8192** (v5 seeds 30..39 + v6 seeds 0..19). If CI-disjoint with margin > v5's, the D=8192 effect is confirmed.
2. **Per-seed paired comparison D=4096 vs D=8192** for seeds 0..19. For each seed, compute Δ_4k and Δ_8k. If MOST seeds have Δ_8k > Δ_4k (especially the negative-at-4k flipping positive-at-8k), that's the substrate-capacity story locked.
3. **Full D-dependence curve**: D ∈ {1024, 2048, 4096, 8192, 16384} with the largest available n at each.

**Decision tree:**
- D=8192 pooled n=30 CI-disjoint with strong per-seed paired signal → Phase 3 graduation at D≥8192; Report 112 frames this as substrate-capacity finding; reopen Phase 5′ with D=8192 substrate as binding.
- D=8192 pooled n=30 CI-overlapping → v5 D=8192 was lucky tail; no graduation; Path γ.
- D=8192 pooled disjoint but per-seed paired comparison mixed → D-dependence weaker than v5 suggested; narrow graduation claim; consider larger n before committing.

In [ ]:
# 1. Clone the repo and apply three patches.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout codex/phase5-prime-bundle-first-scene-memory
!git log --oneline -3

import subprocess
pre_pull = subprocess.run(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py'], capture_output=True, text=True).stdout.strip()
pre_gram = subprocess.run(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py'], capture_output=True, text=True).stdout.strip()
pre_wt   = subprocess.run(['grep', '-c', 'Salesforce/wikitext', 'src/energy_memory/phase2/corpus.py'], capture_output=True, text=True).stdout.strip()
print(f'pre-patch tracers: --lr-pull = {pre_pull}; kernel-trick gram = {pre_gram}; Salesforce/wikitext = {pre_wt}')

patch = r'''diff --git a/experiments/c3_phase3_exit_criterion.py b/experiments/c3_phase3_exit_criterion.py
--- a/experiments/c3_phase3_exit_criterion.py
+++ b/experiments/c3_phase3_exit_criterion.py
@@ -576,6 +576,8 @@ def _run_single_seed_condition(
     k: int,
     alpha_anti: float,
     repulsion_step_size: float,
+    lr_pull: float,
+    lr_push: float,
     device: str,
     repo_root: Path,
     wikitext_corpus: Optional[_WikiTextCorpus] = None,
@@ -756,6 +758,8 @@ def _run_single_seed_condition(
             vocab_size=vocab_size,
             n_events=n_consolidation_events,
             device=device,
+            lr_pull=lr_pull,
+            lr_push=lr_push,
             repulsion_step_size=repulsion_step_size,
         )
 
@@ -838,6 +842,8 @@ def run(
     n_consolidation_events: int = 1000,
     alpha_anti: float = 0.0,
     repulsion_step_size: float = 0.0,
+    lr_pull: float = 0.1,
+    lr_push: float = 0.05,
     device: str,
     output_dir: Path,
     repo_root: Path,
@@ -919,6 +925,8 @@ def run(
                     k=k,
                     alpha_anti=alpha_anti,
                     repulsion_step_size=repulsion_step_size,
+                    lr_pull=lr_pull,
+                    lr_push=lr_push,
                     device=device,
                     repo_root=repo_root,
                     wikitext_corpus=wikitext_corpus,
@@ -1010,6 +1018,8 @@ def run(
             "substrate_repulsion_active": bool(
                 alpha_anti > 0.0 and repulsion_step_size > 0.0
             ),
+            "lr_pull": float(lr_pull),
+            "lr_push": float(lr_push),
             "operating_point": {
                 "D": D,
                 "landscape_size": landscape_size,
@@ -1370,6 +1380,27 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
             "smoke (no inter-atom-separability force)."
         ),
     )
+    parser.add_argument(
+        "--lr-pull",
+        type=float,
+        default=0.1,
+        help=(
+            "Per-event consolidation pull learning rate (OnlineCodebookUpdater "
+            "lr_pull). Default 0.1 matches the existing Path α smoke. Sweep "
+            "above this to test whether consolidation strength is too weak "
+            "to express corpus-specific learning at the synthetic operating "
+            "point."
+        ),
+    )
+    parser.add_argument(
+        "--lr-push",
+        type=float,
+        default=0.05,
+        help=(
+            "Per-event consolidation push learning rate (OnlineCodebookUpdater "
+            "lr_push). Default 0.05 matches the existing Path α smoke."
+        ),
+    )
     parser.add_argument(
         "--repulsion-step-size",
         type=float,
@@ -1468,6 +1499,8 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
         n_consolidation_events=args.n_consolidation_events,
         alpha_anti=args.alpha_anti,
         repulsion_step_size=args.repulsion_step_size,
+        lr_pull=args.lr_pull,
+        lr_push=args.lr_push,
         device=args.device,
         output_dir=output_dir,
         repo_root=repo_root,
diff --git a/src/energy_memory/phase4/consolidation.py b/src/energy_memory/phase4/consolidation.py
--- a/src/energy_memory/phase4/consolidation.py
+++ b/src/energy_memory/phase4/consolidation.py
@@ -639,10 +639,29 @@ class ConsolidationState:
         # Hermitian Gram of centered basin members. For complex (FHRR)
         # tensors, diffs.conj().T @ diffs is Hermitian → real eigenvalues
         # via torch.linalg.eigh.
-        sigma = (diffs.conj().transpose(-1, -2) @ diffs) / float(n)
+        # Compute the eigenvalues of σ = diffs.conj().T @ diffs / n via the
+        # n×n Gram matrix gram = diffs @ diffs.conj().T / n instead of the
+        # D×D scatter matrix. The two matrices share exactly the same set
+        # of non-zero eigenvalues (standard "kernel trick" identity); the
+        # D×D form additionally carries (D - n) trivial zero eigenvalues
+        # because rank(σ) ≤ n_members ≤ basin_trace_buffer_size (64) ≪ D
+        # (4096 by default in this project). That (D - n) zero subspace
+        # makes σ numerically ill-conditioned at the precision available
+        # to torch.linalg.eigvalsh — observed on Colab CUDA at 2026-05-27
+        # as LinAlgError 4095 and even on CPU LAPACK as LinAlgError 5/12.
+        # The n×n Gram path is full-rank for non-degenerate samples and
+        # an order of magnitude smaller (4 KB vs 16 MB at D=4096, n=8).
+        # Mathematically byte-identical at the λ_1 / λ_2 layer used below;
+        # the C.2.2 dynamic's behavior is unchanged.
+        gram = (diffs @ diffs.conj().transpose(-1, -2)) / float(n)
         # Eigh returns ascending eigenvalues. Take top two: λ_1 (last),
         # λ_2 (second-to-last). All ops stay on-device.
-        eigvals = torch.linalg.eigvalsh(sigma)
+        try:
+            eigvals = torch.linalg.eigvalsh(gram)
+        except torch._C._LinAlgError:
+            # Defensive: keep the CPU fallback in case some pathological
+            # input still trips cuSOLVER (e.g. identical basin members).
+            eigvals = torch.linalg.eigvalsh(gram.cpu()).to(gram.device)
         lam_1 = eigvals[-1]
         lam_2 = eigvals[-2] if eigvals.shape[0] >= 2 else torch.zeros_like(lam_1)
         # Clamp at 0 — eigh may return tiny negatives for near-singular Σ.
diff --git a/src/energy_memory/phase2/corpus.py b/src/energy_memory/phase2/corpus.py
--- a/src/energy_memory/phase2/corpus.py
+++ b/src/energy_memory/phase2/corpus.py
@@ -113,7 +113,13 @@ def load_repo_sample_splits(repo_root: Path) -> Dict[str, List[str]]:
 def load_wikitext_splits(name: str = "wikitext-2-raw-v1") -> Dict[str, List[str]]:
     if load_dataset is None:  # pragma: no cover - exercised only when dependency missing
         raise ModuleNotFoundError("datasets is required to load WikiText-2")
-    dataset = load_dataset("wikitext", name)
+    # Use the canonical Salesforce/wikitext namespace. The bare "wikitext"
+    # form worked with older HF stacks but recent huggingface_hub versions
+    # (~0.30+) ship a stricter HF URI parser that rejects any repo id
+    # without an explicit namespace, raising HfUriError. The Salesforce
+    # mirror is the current canonical home of the dataset; config names
+    # ("wikitext-2-raw-v1", "wikitext-103-raw-v1", ...) are unchanged.
+    dataset = load_dataset("Salesforce/wikitext", name)
     return {
         "train": [row["text"] for row in dataset["train"]],
         "validation": [row["text"] for row in dataset["validation"]],
'''

with open('/tmp/c3_combined.patch', 'w') as f:
    f.write(patch)
check = subprocess.run(['git', 'apply', '--check', '/tmp/c3_combined.patch'], capture_output=True, text=True)
if check.returncode == 0:
    subprocess.check_call(['git', 'apply', '/tmp/c3_combined.patch'])
    print('combined patch applied.')
else:
    if int(pre_pull or '0') >= 1 and int(pre_gram or '0') >= 1 and int(pre_wt or '0') >= 1:
        print('all three patches already in branch — skipping apply.')
    else:
        print('PATCH APPLY FAILED:'); print(check.stderr)
        raise SystemExit('Cannot continue.')

post_pull = subprocess.check_output(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py']).decode().strip()
post_gram = subprocess.check_output(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py']).decode().strip()
post_wt   = subprocess.check_output(['grep', '-c', 'Salesforce/wikitext', 'src/energy_memory/phase2/corpus.py']).decode().strip()
print(f'post-patch tracers: --lr-pull = {post_pull}; kernel-trick = {post_gram}; Salesforce/wikitext = {post_wt}')
assert int(post_pull) >= 1 and int(post_gram) >= 1 and int(post_wt) >= 1, 'patches missing'

In [ ]:
# 2. Mount Drive + verify historical JSONs available for aggregation.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuro-ai/results', exist_ok=True)
from pathlib import Path
v3_dir = Path('/content/drive/MyDrive/neuro-ai/results/c3_followup_v3_2026-05-27')
v4_dir = Path('/content/drive/MyDrive/neuro-ai/results/c3_robustness_v4_2026-05-27')
v5_dir = Path('/content/drive/MyDrive/neuro-ai/results/c3_v5_2026-05-27')
v3_n10  = list(v3_dir.glob('n10_wikitext_base_seed*')) if v3_dir.exists() else []
v4_rep  = list(v4_dir.glob('replicate_seeds10_19_seed*')) if v4_dir.exists() else []
v5_d4   = list(v5_dir.glob('Dsweep_D4096_seed*')) if v5_dir.exists() else []
v5_d8   = list(v5_dir.glob('Dsweep_D8192_seed*')) if v5_dir.exists() else []
print('historical data check:')
print(f'  v3 D=4096 seeds 0..9       : {len(v3_n10)}/10')
print(f'  v4 D=4096 seeds 10..19     : {len(v4_rep)}/10')
print(f'  v5 D=4096 seeds 30..39     : {len(v5_d4)}/10')
print(f'  v5 D=8192 seeds 30..39     : {len(v5_d8)}/10  (NEEDED for D=8192 pooled n=30)')

In [ ]:
# 3. Install deps.
!pip install -q "datasets<3"
import sys, torch, numpy as np, datasets
print(f'python: {sys.version.split()[0]} | torch: {torch.__version__} | numpy: {np.__version__} | datasets: {datasets.__version__}')
print(f'cuda available: {torch.cuda.is_available()}; device 0: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# 4. Pre-warm wikitext-2 HF cache.
import sys; sys.path.insert(0, '/content/Neuro-AI/src')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
print('warming wikitext-2-raw-v1 cache...')
splits = load_corpus_splits('wikitext', Path('/content/Neuro-AI'), wikitext_name='wikitext-2-raw-v1')
print(f'  train: {len(splits["train"])} rows; val: {len(splits["validation"])} rows; test: {len(splits["test"])} rows')
del splits; import gc; gc.collect()

In [ ]:
# 5. GPU info.
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 6. SMOKE — confirm runtime sane at the LARGEST D (D=16384) since it's the most likely to break.
import subprocess, sys, os
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
smoke_out = Path('reports/c3_v6_smoke_2026-05-27')
smoke_out.mkdir(parents=True, exist_ok=True)
smoke_log = Path('reports/c3_v6_smoke.log')
cmd = [sys.executable, 'experiments/c3_phase3_exit_criterion.py',
       '--seeds', '0', '--device', 'cuda',
       '--D', '16384',
       '--lr-pull', '0.1', '--lr-push', '0.05',
       '--n-consolidation-events', '100',
       '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
       '--output-dir', str(smoke_out)]
with smoke_log.open('w') as logf:
    rc = subprocess.call(cmd, stdout=logf, stderr=subprocess.STDOUT)
print(f'smoke exit code: {rc}; json: {(smoke_out / "c3_summary.json").exists()}')
print('\n=== smoke log (last 30 lines) ===')
!tail -30 {smoke_log}
if rc != 0:
    raise SystemExit('Smoke failed — abort.')
print('\nSmoke OK.')

In [ ]:
# 7. PARALLEL launch — D=8192 seeds 0..19 (20 procs) + D=16384 seeds 30..39 (10 procs) = 30 subprocesses.
import subprocess, os, time, signal, sys
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
PY = sys.executable

PROBES = [
    # (tag, D, seeds_list)
    ('D8192_confirm_seeds0_19', 8192,  list(range(0, 20))),
    ('D16384_seeds30_39',       16384, list(range(30, 40))),
]

ENTRIES = []
for tag, D, seeds in PROBES:
    for seed in seeds:
        ENTRIES.append((tag, D, seed))
print(f'launching {len(ENTRIES)} per-seed subprocesses ({len(PROBES)} probes)')
for tag, D, seeds in PROBES:
    print(f'  {tag}: D={D}, seeds={seeds[0]}..{seeds[-1]} ({len(seeds)} seeds)')

log_root = Path('reports/c3_v6_logs')
log_root.mkdir(parents=True, exist_ok=True)

def out_dir_for(tag, seed):
    return f'reports/c3_v6_{tag}_seed{seed}_2026-05-27'

def launch(tag, D, seed):
    out_dir = out_dir_for(tag, seed)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'{tag}_seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [PY, 'experiments/c3_phase3_exit_criterion.py',
           '--seeds', str(seed), '--device', 'cuda',
           '--D', str(D),
           '--lr-pull', '0.1', '--lr-push', '0.05',
           '--n-consolidation-events', '1000',
           '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
           '--corpus-source', 'wikitext',
           '--output-dir', out_dir]
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
    return proc, logf, out_dir, log_path

def snapshot(remaining, total, t0):
    elapsed = (time.time() - t0) / 60
    n_done = total - len(remaining)
    print(f'  --- snapshot at {elapsed:.1f} min — {n_done}/{total} done, {len(remaining)} running ---')
    try:
        gpu = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        print(f'  GPU: {gpu}')
    except Exception as e:
        print(f'  GPU snapshot failed: {e}')
    from collections import Counter
    cond_running = Counter()
    for key in remaining:
        cond_running[key.rsplit('_seed', 1)[0]] += 1
    for tag, D, seeds in PROBES:
        n_run = cond_running.get(tag, 0)
        n_done_tag = len(seeds) - n_run
        print(f'    {tag:>26}: {n_done_tag}/{len(seeds)} done')

def kill_all(remaining):
    for key, (proc, logf, _, _) in remaining.items():
        try:
            proc.send_signal(signal.SIGKILL); logf.close()
        except Exception:
            pass

# Stagger launches. Launching the bigger D=16384 procs FIRST so they have head start.
ENTRIES_ORDERED = sorted(ENTRIES, key=lambda e: -e[1])  # bigger D first
procs = {}
for entry in ENTRIES_ORDERED:
    tag, D, seed = entry
    key = f'{tag}_seed{seed}'
    procs[key] = launch(*entry)
    time.sleep(1.5)
print(f'all {len(procs)} cells launched  ({time.strftime("%H:%M:%S")})')

t0 = time.time()
remaining = dict(procs)
total = len(procs)
failures = []
poll_count = 0
try:
    while remaining:
        done_this_round = []
        for key, (proc, logf, out_dir, log_path) in remaining.items():
            rc = proc.poll()
            if rc is not None:
                logf.close()
                elapsed = (time.time() - t0) / 60
                json_exists = Path(out_dir, 'c3_summary.json').exists()
                ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                print(f'  [{elapsed:5.1f} min] {key:>34}: {ok}  json={json_exists}')
                if rc != 0:
                    failures.append(key)
                    print(f'    --- last 30 lines of {log_path} ---')
                    try:
                        out = subprocess.check_output(['tail', '-30', str(log_path)],
                            stderr=subprocess.DEVNULL).decode()
                        for line in out.splitlines():
                            print(f'    | {line}')
                    except Exception as e:
                        print(f'    | (could not read log: {e})')
                    print('    --- end log ---')
                done_this_round.append(key)
        for key in done_this_round:
            del remaining[key]
        if remaining:
            poll_count += 1
            if poll_count % 3 == 0:
                snapshot(remaining, total, t0)
            time.sleep(30)
except KeyboardInterrupt:
    print('\n!!! Interrupted !!!')
    kill_all(remaining); raise

print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')
print(f'failures: {len(failures)}/{total}')
if failures:
    print('  failed keys:', failures)
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 7b. EMERGENCY kill.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'c3_phase3_exit_criterion' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL); print(f'  killed {pid}'); killed += 1
        except Exception as e:
            print(f'  err: {e}')
print(f'killed {killed} workers')

In [ ]:
# 8. Copy v6 results + logs to Drive.
import shutil, os
dst_root = '/content/drive/MyDrive/neuro-ai/results/c3_v6_2026-05-27'
os.makedirs(dst_root, exist_ok=True)
PROBES = [
    ('D8192_confirm_seeds0_19', list(range(0, 20))),
    ('D16384_seeds30_39',       list(range(30, 40))),
]
count = 0
for tag, seeds in PROBES:
    for seed in seeds:
        src = f'reports/c3_v6_{tag}_seed{seed}_2026-05-27'
        if os.path.isdir(src):
            shutil.copytree(src, f'{dst_root}/{tag}_seed{seed}', dirs_exist_ok=True)
            count += 1
if os.path.isdir('reports/c3_v6_logs'):
    shutil.copytree('reports/c3_v6_logs', f'{dst_root}/colab_logs', dirs_exist_ok=True)
print(f'copied {count} per-seed dirs + logs to {dst_root}')
!ls {dst_root} | head -20

In [ ]:
# 9. AGGREGATION — three outputs:
#    (1) D=8192 pooled n=30 (v5 seeds 30..39 + v6 seeds 0..19) → confirmation test.
#    (2) Per-seed paired comparison D=4096 vs D=8192 for seeds 0..19 (substrate-capacity test).
#    (3) Full D-dependence curve with the best-available n at each D.
import json, math
from pathlib import Path

STRATA = ('tight', 'spread', 'borderline')

def wilson(s, t, z=1.96):
    if t == 0: return (0.0, 0.0, 0.0)
    p = s / t
    denom = 1 + z*z/t
    center = (p + z*z/(2*t)) / denom
    half = (z * math.sqrt(p*(1-p)/t + z*z/(4*t*t))) / denom
    return (p, max(0.0, center - half), min(1.0, center + half))

def read_rows_from_dir(parent_dir, glob_pattern):
    p = Path(parent_dir)
    if not p.exists():
        return [], 0, {}
    rows = []
    seed_count = 0
    per_seed_rows = {}  # seed → list of per_cell rows
    for subdir in sorted(p.glob(glob_pattern)):
        json_path = subdir / 'c3_summary.json'
        if not json_path.exists():
            continue
        d = json.loads(json_path.read_text())
        for r in d['per_cell_rows']:
            rows.append(r)
            seed = int(r['seed'])
            per_seed_rows.setdefault(seed, []).append(r)
        seed_count += 1
    return rows, seed_count, per_seed_rows

def aggregate_rows(rows):
    agg = {}
    modes = sorted({r['theta_prime_mode'] for r in rows})
    for mode in modes:
        agg[mode] = {}
        for is_control in (False, True):
            key = 'shuffled_control' if is_control else 'standard'
            agg[mode][key] = {}
            for stratum in STRATA:
                tot_s = 0; tot_t = 0
                for r in rows:
                    if r['theta_prime_mode'] != mode or r['is_control'] != is_control: continue
                    cell = r['per_stratum'][stratum]
                    tot_s += int(cell['successes']); tot_t += int(cell['trials'])
                p, lo, hi = wilson(tot_s, tot_t)
                agg[mode][key][stratum] = {'successes': tot_s, 'trials': tot_t,
                    'recall_at_k': p, 'wilson_lower': lo, 'wilson_upper': hi}
        agg[mode]['delta_standard_minus_control'] = {}
        for stratum in STRATA:
            s = agg[mode]['standard'][stratum]; c = agg[mode]['shuffled_control'][stratum]
            agg[mode]['delta_standard_minus_control'][stratum] = {
                'delta_recall_at_k': s['recall_at_k'] - c['recall_at_k'],
                'standard_trials': s['trials'], 'control_trials': c['trials'],
                'ci_disjoint_standard_beats_control': s['wilson_lower'] > c['wilson_upper']}
    return agg, modes

def print_agg(label, agg, modes, n_seeds):
    print(f'\n--- {label} (n_seeds={n_seeds}) ---')
    for mode in modes:
        for stratum in STRATA:
            s = agg[mode]['standard'][stratum]; c = agg[mode]['shuffled_control'][stratum]
            dl = agg[mode]['delta_standard_minus_control'][stratum]
            if s['trials'] == 0 and c['trials'] == 0: continue
            std_str  = f'{s["recall_at_k"]:.3f} [{s["wilson_lower"]:.3f},{s["wilson_upper"]:.3f}]'
            ctrl_str = f'{c["recall_at_k"]:.3f} [{c["wilson_lower"]:.3f},{c["wilson_upper"]:.3f}]'
            disj = 'YES' if dl['ci_disjoint_standard_beats_control'] else 'no'
            print(f'  {mode:>11} {stratum:>11}  std {std_str:>22}  ctrl {ctrl_str:>22}  '
                  f'Δ {dl["delta_recall_at_k"]:>+7.3f}  disjoint? {disj}  n_std={s["trials"]:>5}  n_ctrl={c["trials"]:>5}')

import os
merged_root = '/content/drive/MyDrive/neuro-ai/results/c3_v6_2026-05-27/_merged'
os.makedirs(merged_root, exist_ok=True)

# ============================================================================
# (1) D=8192 POOLED n=30: confirmation test
# ============================================================================
print('=' * 100)
print('(1) D=8192 POOLED n=30 (v5 seeds 30..39 + v6 seeds 0..19) — confirmation test')
print('=' * 100)
v5_d8_rows, v5_d8_n, _ = read_rows_from_dir(
    '/content/drive/MyDrive/neuro-ai/results/c3_v5_2026-05-27', 'Dsweep_D8192_seed*')
v6_d8_rows, v6_d8_n, _ = read_rows_from_dir('reports', 'c3_v6_D8192_confirm_seeds0_19_seed*_2026-05-27')
print(f'  v5 D=8192 seeds 30..39: {v5_d8_n} seeds')
print(f'  v6 D=8192 seeds 0..19:  {v6_d8_n} seeds')
pool_d8_rows = v5_d8_rows + v6_d8_rows
total_d8 = v5_d8_n + v6_d8_n
if pool_d8_rows:
    agg_d8, modes_d8 = aggregate_rows(pool_d8_rows)
    print_agg(f'D=8192 POOLED n={total_d8}', agg_d8, modes_d8, total_d8)
    with open(f'{merged_root}/D8192_pooled_n{total_d8}.json', 'w') as f:
        json.dump({'D': 8192, 'n_seeds': total_d8, 'aggregated': agg_d8}, f, indent=2)
else:
    print('  no D=8192 data — aggregation skipped.')

# ============================================================================
# (2) PER-SEED PAIRED D=4096 vs D=8192 for seeds 0..19
# ============================================================================
print('\n' + '=' * 100)
print('(2) PER-SEED PAIRED D=4096 vs D=8192 for seeds 0..19 (substrate-capacity test)')
print('=' * 100)
# D=4096 rows for seeds 0..19: v3 (0..9) + v4 (10..19)
v3_rows, v3_n, v3_per_seed = read_rows_from_dir(
    '/content/drive/MyDrive/neuro-ai/results/c3_followup_v3_2026-05-27', 'n10_wikitext_base_seed*')
v4_rows, v4_n, v4_per_seed = read_rows_from_dir(
    '/content/drive/MyDrive/neuro-ai/results/c3_robustness_v4_2026-05-27', 'replicate_seeds10_19_seed*')
_, _, v6_d8_per_seed = read_rows_from_dir('reports', 'c3_v6_D8192_confirm_seeds0_19_seed*_2026-05-27')

d4_per_seed = {**v3_per_seed, **v4_per_seed}  # seeds 0..19
d8_per_seed = v6_d8_per_seed  # seeds 0..19

def per_seed_delta(rows, mode, stratum):
    """For one seed's per_cell rows, return Δ = std - ctrl for (mode, stratum)."""
    std_s = std_t = ctrl_s = ctrl_t = 0
    for r in rows:
        if r['theta_prime_mode'] != mode: continue
        cell = r['per_stratum'][stratum]
        if r['is_control']:
            ctrl_s += int(cell['successes']); ctrl_t += int(cell['trials'])
        else:
            std_s += int(cell['successes']); std_t += int(cell['trials'])
    if std_t == 0 or ctrl_t == 0:
        return None
    return (std_s/std_t) - (ctrl_s/ctrl_t)

MODE = 'default'; STRATUM = 'spread'  # the primary v3-disjoint metric
print(f'\nPer-seed Δ for ({MODE}/{STRATUM}):')
print(f'{"seed":>5} {"Δ_4096":>10} {"Δ_8192":>10} {"shift":>10} {"sign_flip?":>12}')
n_flipped_neg_to_pos = 0
n_total = 0
n_8k_gt_4k = 0
n_8k_positive = 0
for seed in range(20):
    d4 = d4_per_seed.get(seed)
    d8 = d8_per_seed.get(seed)
    if d4 is None or d8 is None:
        print(f'{seed:>5}  (missing)')
        continue
    dl4 = per_seed_delta(d4, MODE, STRATUM)
    dl8 = per_seed_delta(d8, MODE, STRATUM)
    if dl4 is None or dl8 is None:
        print(f'{seed:>5}  (no data for stratum)')
        continue
    n_total += 1
    shift = dl8 - dl4
    flip = (dl4 < 0 and dl8 > 0)
    if flip: n_flipped_neg_to_pos += 1
    if dl8 > dl4: n_8k_gt_4k += 1
    if dl8 > 0: n_8k_positive += 1
    print(f'{seed:>5} {dl4:>+10.4f} {dl8:>+10.4f} {shift:>+10.4f} {"flip↑" if flip else "":>12}')
print()
print(f'  seeds with Δ_8192 > Δ_4096: {n_8k_gt_4k}/{n_total}')
print(f'  seeds with Δ_8192 > 0:      {n_8k_positive}/{n_total}')
print(f'  seeds with neg→pos flip:    {n_flipped_neg_to_pos}/{n_total}')

# ============================================================================
# (3) FULL D-DEPENDENCE CURVE
# ============================================================================
print('\n' + '=' * 100)
print('(3) FULL D-DEPENDENCE CURVE (largest available n at each D)')
print('=' * 100)
# Build per-D pooled estimates.
v5_d1_rows, v5_d1_n, _ = read_rows_from_dir(
    '/content/drive/MyDrive/neuro-ai/results/c3_v5_2026-05-27', 'Dsweep_D1024_seed*')
v5_d2_rows, v5_d2_n, _ = read_rows_from_dir(
    '/content/drive/MyDrive/neuro-ai/results/c3_v5_2026-05-27', 'Dsweep_D2048_seed*')
v5_d4_rows, v5_d4_n, _ = read_rows_from_dir(
    '/content/drive/MyDrive/neuro-ai/results/c3_v5_2026-05-27', 'Dsweep_D4096_seed*')
v6_d16_rows, v6_d16_n, _ = read_rows_from_dir('reports', 'c3_v6_D16384_seeds30_39_seed*_2026-05-27')

d4_pool_rows = v3_rows + v4_rows + v5_d4_rows  # n≈30
d4_pool_n    = v3_n + v4_n + v5_d4_n

D_DATA = [
    (1024,  v5_d1_rows,   v5_d1_n),
    (2048,  v5_d2_rows,   v5_d2_n),
    (4096,  d4_pool_rows, d4_pool_n),
    (8192,  pool_d8_rows, total_d8),
    (16384, v6_d16_rows,  v6_d16_n),
]
print(f'\n{MODE}/{STRATUM} (primary v3 disjoint metric):')
print(f'{"D":>6} {"n":>4} {"std R@K":>22} {"ctrl R@K":>22} {"Δ":>8} {"disjoint?":>9}')
for D, rows, n in D_DATA:
    if n == 0:
        print(f'{D:>6} {n:>4}  MISSING')
        continue
    agg, _ = aggregate_rows(rows)
    if MODE not in agg: continue
    s = agg[MODE]['standard'][STRATUM]; c = agg[MODE]['shuffled_control'][STRATUM]
    dl = agg[MODE]['delta_standard_minus_control'][STRATUM]
    std_str  = f'{s["recall_at_k"]:.3f} [{s["wilson_lower"]:.3f},{s["wilson_upper"]:.3f}]'
    ctrl_str = f'{c["recall_at_k"]:.3f} [{c["wilson_lower"]:.3f},{c["wilson_upper"]:.3f}]'
    disj = 'YES' if dl['ci_disjoint_standard_beats_control'] else 'no'
    print(f'{D:>6} {n:>4} {std_str:>22} {ctrl_str:>22} {dl["delta_recall_at_k"]:>+8.3f} {disj:>9}')

# Calibrated/tight too — the OTHER v3-disjoint metric.
MODE2 = 'calibrated'; STRATUM2 = 'tight'
print(f'\n{MODE2}/{STRATUM2} (the other v3 disjoint metric):')
print(f'{"D":>6} {"n":>4} {"std R@K":>22} {"ctrl R@K":>22} {"Δ":>8} {"disjoint?":>9}')
for D, rows, n in D_DATA:
    if n == 0: continue
    agg, _ = aggregate_rows(rows)
    if MODE2 not in agg: continue
    s = agg[MODE2]['standard'][STRATUM2]; c = agg[MODE2]['shuffled_control'][STRATUM2]
    dl = agg[MODE2]['delta_standard_minus_control'][STRATUM2]
    if s['trials'] == 0: continue
    std_str  = f'{s["recall_at_k"]:.3f} [{s["wilson_lower"]:.3f},{s["wilson_upper"]:.3f}]'
    ctrl_str = f'{c["recall_at_k"]:.3f} [{c["wilson_lower"]:.3f},{c["wilson_upper"]:.3f}]'
    disj = 'YES' if dl['ci_disjoint_standard_beats_control'] else 'no'
    print(f'{D:>6} {n:>4} {std_str:>22} {ctrl_str:>22} {dl["delta_recall_at_k"]:>+8.3f} {disj:>9}')

# ============================================================================
# VERDICT
# ============================================================================
print('\n' + '=' * 100)
print('VERDICT')
print('=' * 100)
if pool_d8_rows:
    agg_d8_default_spread = agg_d8['default']['delta_standard_minus_control']['spread']
    d8_disjoint_def = agg_d8_default_spread['ci_disjoint_standard_beats_control']
    d8_delta_def = agg_d8_default_spread['delta_recall_at_k']
    agg_d8_cal_tight = agg_d8['calibrated']['delta_standard_minus_control']['tight']
    d8_disjoint_cal = agg_d8_cal_tight['ci_disjoint_standard_beats_control']
    d8_delta_cal = agg_d8_cal_tight['delta_recall_at_k']
    print(f'  D=8192 n={total_d8} default/spread:    Δ={d8_delta_def:+.3f}  disjoint={d8_disjoint_def}')
    print(f'  D=8192 n={total_d8} calibrated/tight:  Δ={d8_delta_cal:+.3f}  disjoint={d8_disjoint_cal}')
    print(f'  Per-seed paired (n={n_total}): {n_8k_gt_4k}/{n_total} have Δ_8192 > Δ_4096; '
          f'{n_8k_positive}/{n_total} have Δ_8192 > 0; {n_flipped_neg_to_pos}/{n_total} flipped neg→pos')
    print()
    if d8_disjoint_def and d8_disjoint_cal and n_8k_gt_4k >= 0.7 * n_total:
        print('CONFIRMED: D=8192 effect is real and substrate-dim-dependent.')
        print('  → Phase 3 graduates at D≥8192 (the original v3 D=4096 was operating-point-borderline).')
        print('  → Write Report 112 framing this as a substrate-capacity finding.')
        print('  → Reopen Phase 5′ with D=8192 substrate as binding.')
    elif d8_disjoint_def or d8_disjoint_cal:
        print('PARTIAL CONFIRMATION: D=8192 effect holds in some strata but per-seed pattern mixed.')
        print('  → Write Report 112 as a substrate-capacity sensitivity study, narrow graduation claim.')
    else:
        print('FALSIFIED: D=8192 v5 result was lucky-tail like v3 was at D=4096.')
        print('  → No graduation; clean Path γ pivot.')
else:
    print('  no D=8192 data — verdict cannot be rendered.')